In [3]:
import requests
from bs4 import BeautifulSoup

# URL for BBC News homepage
news_url = "https://inu.edu.pk/computer-science-department/"

# Fetch and parse the page
response = requests.get(news_url)
news_soup = BeautifulSoup(response.content, "html.parser")

# Try multiple selectors for headlines
headlines = news_soup.find_all("h3", class_="gs-c-promo-heading__title")

# If not found, try anchor tags with class 'gs-c-promo-heading'
if not headlines:
    promo_anchors = news_soup.select("a.gs-c-promo-heading")
    headlines = [a for a in promo_anchors if a.text.strip()]

# If still not found, fallback to all anchor tags with '/news/' in href and non-empty text
if not headlines:
    headlines = [
        a for a in news_soup.find_all("a", href=True)
        if "/computer-science-department/" in a["href"] and a.text.strip()
    ]

if not headlines:
    print("No headlines found using known selectors.")
else:
    for idx, headline in enumerate(headlines, start=1):
        # Get the headline text
        headline_text = headline.text.strip()
        print(f"{idx}. {headline_text}")

1. Computer Science Department
2. Computer Science Department
3. Computer Science Department
4. Computer Science Department


## **Data Scraping code with markdown file**

In [5]:
# save the headlines, links, and snippets to a markdown file
from datetime import datetime
import requests
from bs4 import BeautifulSoup

# URL for BBC News homepage
news_url = "https://www.bbc.com/business"

# Fetch and parse the page
response = requests.get(news_url)
news_soup = BeautifulSoup(response.content, "html.parser")

# Try multiple selectors for headlines
headlines = news_soup.find_all("h3", class_="gs-c-promo-heading__title")

# If not found, try anchor tags with class 'gs-c-promo-heading'
if not headlines:
    promo_anchors = news_soup.select("a.gs-c-promo-heading")
    headlines = [a for a in promo_anchors if a.text.strip()]

# If still not found, fallback to all anchor tags with '/news/' in href and non-empty text
if not headlines:
    headlines = [
        a for a in news_soup.find_all("a", href=True)
        if "/business/" in a["href"] and a.text.strip()
    ]

if not headlines:
    print("No headlines found using known selectors.")
else:
    # Get current date and time for filename
    now_str = datetime.now().strftime("%Y%m%d_%H%M%S")
    md_filename = f"bbc_headlines_{now_str}.md"

    with open(md_filename, "w") as f:
        for idx, headline in enumerate(headlines, start=1):
            headline_text = headline.text.strip()
            link = None
            if headline.name == "a" and headline.has_attr("href"):
                link = headline["href"]
            else:
                parent_a = headline.find_parent("a", href=True)
                if parent_a:
                    link = parent_a["href"]
            if link and link.startswith("/"):
                link = "https://www.bbc.com" + link

            # Fetch the news article page to get date and snippet
            date_str = ""
            snippet = ""
            if link:
                try:
                    article_resp = requests.get(link)
                    article_soup = BeautifulSoup(article_resp.content, "html.parser")
                    # Extract paragraphs for snippet
                    article_tag = article_soup.find("article")
                    if not article_tag:
                        article_tag = article_soup.find(attrs={"role": "main"})
                    if article_tag:
                        paragraphs = article_tag.find_all("p")
                    else:
                        paragraphs = article_soup.find_all("p")
                    article_text = " ".join([p.get_text(strip=True) for p in paragraphs])
                    snippet = article_text[:400] + ("..." if len(article_text) > 400 else "")
                    # Extract date
                    time_tag = article_soup.find("time")
                    if not time_tag:
                        meta_time = article_soup.find("meta", attrs={"property": "article:published_time"})
                        if meta_time and meta_time.has_attr("content"):
                            date_str = meta_time["content"]
                    if not date_str and time_tag and time_tag.has_attr("datetime"):
                        date_str = time_tag["datetime"]
                    elif not date_str and time_tag:
                        date_str = time_tag.get_text(strip=True)
                    if date_str:
                        try:
                            dt = datetime.fromisoformat(date_str.replace("Z", "+00:00"))
                            date_str = dt.strftime("%Y-%m-%d %H:%M:%S %Z")
                        except Exception:
                            pass
                except Exception:
                    snippet = "(Could not fetch article)"
                    date_str = ""
            f.write(f"{idx}. [{headline_text}]({link})\n")
            f.write(f"   Date: {date_str if date_str else '(No date found)'}\n")
            f.write(f"   News: {snippet}\n")


# Scraping Headings and Paragraphs from INU Computer Science Department
This notebook demonstrates how to scrape all headings (h1-h4) and paragraphs from the INU Computer Science Department page.

In [4]:
import requests
from bs4 import BeautifulSoup

url = "https://inu.edu.pk/computer-science-department/"
response = requests.get(url)
soup = BeautifulSoup(response.content, "html.parser")

# Extract all headings (h1-h4)
headings = []
for tag in ['h1', 'h2', 'h3', 'h4']:
    for h in soup.find_all(tag):
        headings.append((tag, h.get_text(strip=True)))

# Extract all paragraphs
paragraphs = [p.get_text(strip=True) for p in soup.find_all('p') if p.get_text(strip=True)]

print("Headings:")
for tag, text in headings:
    print(f"{tag.upper()}: {text}")

print("\nParagraphs:")
for idx, para in enumerate(paragraphs, 1):
    print(f"{idx}. {para}")

Headings:
H1: COMPUTER SCIENCE DEPARTMENT
H3: BROWSE LINK
H3: ABOUT DEPARTEMENT
H3: VISION
H3: MISSION

Paragraphs:
1. The objective of this school is to produce computer scientists, who from the backbone of the rapidly growing IT industry. The department is focused on developing an in depth understanding of both the theoretical and practical aspects of computer science through rigorous course work.
2. Students are given unique opportunities to go beyond traditional learning and get involved in research activities through various research and industrial collaboration programs carried out on campus.
3. Our courseware is tailored according to the international standards to nurture capacity building and original thinking in our graduates for life-long-learning. Our graduates would be highly sought after by both national and international IT industry.
4. To contribute towards highest quality educational needs of students within the core areas of Computer Science distinguished by its impact

---
You can further process or save the scraped data as needed (e.g., to a CSV or text file).

# Scrape Headings and Their Content from INU Computer Science Department
This notebook scrapes each heading (h1-h4) and the paragraphs or text that follow it, preserving the structure as seen on the website.

In [5]:
import requests
from bs4 import BeautifulSoup, Tag

url = "https://inu.edu.pk/computer-science-department/"
response = requests.get(url)
soup = BeautifulSoup(response.content, "html.parser")

results = []
current_heading = None
current_content = []

# Go through all elements in the body in order
for elem in soup.body.descendants:
    if isinstance(elem, Tag):
        if elem.name in ["h1", "h2", "h3", "h4"]:
            if current_heading:
                results.append((current_heading, current_content))
            current_heading = elem.get_text(strip=True)
            current_content = []
        elif elem.name == "p" and current_heading:
            text = elem.get_text(strip=True)
            if text:
                current_content.append(text)
# Add the last heading and its content
if current_heading:
    results.append((current_heading, current_content))

# Print the structured output
for idx, (heading, paras) in enumerate(results, 1):
    print(f"{idx}. {heading}")
    for p in paras:
        print(f"   {p}")
    print()

1. COMPUTER SCIENCE DEPARTMENT

2. BROWSE LINK

3. ABOUT DEPARTEMENT
   The objective of this school is to produce computer scientists, who from the backbone of the rapidly growing IT industry. The department is focused on developing an in depth understanding of both the theoretical and practical aspects of computer science through rigorous course work.
   Students are given unique opportunities to go beyond traditional learning and get involved in research activities through various research and industrial collaboration programs carried out on campus.
   Our courseware is tailored according to the international standards to nurture capacity building and original thinking in our graduates for life-long-learning. Our graduates would be highly sought after by both national and international IT industry.

4. VISION
   To contribute towards highest quality educational needs of students within the core areas of Computer Science distinguished by its impact on academia, industry and society.


---
This approach preserves the structure: each heading is followed by its related paragraphs, similar to the website layout.